## Imports

In [2]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    import gdown
    drive.mount('/content/drive')

In [ ]:
import io
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from pathlib import Path

## Setup
1) Loading data
2) Constants

In [ ]:
# 1)
import os
import sys

# Resolve data paths robustly for both interactive and nbconvert/SLURM execution
candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/home/uoftshen/scratch/CHE1148_Team_Protein_Origami'),
]

train_path = None
val_path = None
for root in candidate_roots:
    t = root / 'data/processed/train_sampled_esm2_650m.pt'
    v = root / 'data/processed/val_full_esm2_650m.pt'
    if t.exists() and v.exists():
        train_path = t
        val_path = v
        break

if train_path is not None and val_path is not None:
    print('Using local processed embeddings:')
    print('  train_path =', train_path)
    print('  val_path   =', val_path)
else:
    if 'google.colab' not in sys.modules:
        raise FileNotFoundError(
            'Local processed .pt files not found in expected locations. '
            'Checked cwd/data/processed and parent/data/processed.'
        )

    import gdown

    # Training file
    train_file_id = '1hT51HoLUSZohAl4CRIRopnh0gqnj8Pz4'
    train_output = 'train_sampled_esm2_650m.pt'
    gdown.download(f'https://drive.google.com/uc?id={train_file_id}', train_output, quiet=False)
    train_path = Path(train_output)

    # Validation file
    val_file_id = '1H0lC_7G0cwr8hB0dMyrWf7tcEUB8oB4e'
    val_output = 'val_full_esm2_650m.pt'
    gdown.download(f'https://drive.google.com/uc?id={val_file_id}', val_output, quiet=False)
    val_path = Path(val_output)

In [4]:
# 2)
# Model Architecture
INPUT_DIM = 1280          # ESM-2 embedding dimension
LATENT_DIM = 256          # Latent space dimensionality
HIDDEN_DIM = 1024         # Hidden layer dimension

BATCH_SIZE = 64
LEARNING_RATE = 1e-4
NUM_EPOCHS = 100
RECONSTRUCTION_WEIGHT = 1.0
KL_WEIGHT = 0.05
PREDICTION_WEIGHT = 1.0
OPTIM_STEPS = 25          # Steps for latent optimization
OPTIM_LR = 0.01           # Learning rate for latent optimization

## Robust Loader

In [5]:
def robust_pt_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except Exception:
        import pickle
        class CPU_Unpickler(pickle.Unpickler):
            def find_class(self, module, name):
                if module == 'torch.storage' and name == '_load_from_bytes':
                    return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
                return super().find_class(module, name)
        with open(path, 'rb') as f:
            return CPU_Unpickler(f).load()

## Dataset Class

In [6]:
class ProteinDataset(Dataset):
    def __init__(self, pt_path, scaler=None):
        print(f"Loading Dataset: {pt_path}")
        data_obj = robust_pt_load(pt_path)

        self.embeddings = torch.from_numpy(np.array(data_obj['seq_embeddings'])).float()
        labels_np = np.array(data_obj['delta_g']).reshape(-1, 1)
        self.clusters = np.array(data_obj['wt_cluster'])
        self.mut_types = np.array(data_obj['mut_type'])

        self.cluster_to_indices = {}
        for idx, c in enumerate(self.clusters):
            if c not in self.cluster_to_indices:
                self.cluster_to_indices[c] = []
            self.cluster_to_indices[c].append(idx)

        self.wt_lookup = {}
        for i in range(len(self.clusters)):
            if str(self.mut_types[i]).lower().strip() == 'wt':
                self.wt_lookup[self.clusters[i]] = {
                    'dg': labels_np[i][0],
                    'emb': self.embeddings[i],
                    'idx': i
                }

        if scaler is None:
            self.scaler = StandardScaler()
            self.labels_scaled = self.scaler.fit_transform(labels_np)
        else:
            self.scaler = scaler
            self.labels_scaled = self.scaler.transform(labels_np)

        self.labels_scaled = torch.from_numpy(self.labels_scaled).float()
        self.raw_labels = torch.from_numpy(labels_np).float()

    def __len__(self):
        return self.embeddings.shape[0]

    def __getitem__(self, idx):
        cluster_id = self.clusters[idx]
        wt_info = self.wt_lookup.get(cluster_id, None)
        wt_dg = wt_info['dg'] if wt_info else self.raw_labels[idx].item()
        wt_emb = wt_info['emb'] if wt_info else self.embeddings[idx]

        return {
            'emb': self.embeddings[idx],
            'label_scaled': self.labels_scaled[idx],
            'wt_dg': torch.tensor([wt_dg]).float(),
            'wt_emb': wt_emb,
            'cluster_id': cluster_id
        }

## Protein VAE Model

In [7]:
class ProteinVAE(nn.Module):
    def __init__(self, input_dim=1280, latent_dim=256, hidden_dim=1024):
        super(ProteinVAE, self).__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim * 2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
        self.regressor = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = h[:, :self.latent_dim], h[:, self.latent_dim:]
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        pred_y = self.regressor(z if self.training else mu)
        return recon, pred_y, mu, logvar

## Optimisation

In [8]:
def optimize_latent(model, emb, steps=25, lr=0.01):
    model.eval()

    with torch.no_grad():
        h = model.encoder(emb)
        z_start = h[:, :model.latent_dim].detach()

    z_opt = z_start.clone().requires_grad_(True)
    optimizer = optim.Adam([z_opt], lr=lr)

    for _ in range(steps):
        optimizer.zero_grad()
        pred_y = model.regressor(z_opt)
        loss = pred_y.mean() + 0.05 * torch.pow(z_opt, 2).mean()
        loss.backward()
        optimizer.step()

    return z_opt.detach()

## Training/Evaluation Pipelines

In [9]:
# --- 5. Training and Evaluation Pipeline ---
def train_and_validate(train_path, val_path, epochs=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    train_ds = ProteinDataset(train_path)
    val_ds = ProteinDataset(val_path, scaler=train_ds.scaler)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    model = ProteinVAE().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            x, y = batch['emb'].to(device), batch['label_scaled'].to(device)
            optimizer.zero_grad()
            recon, pred_y, mu, logvar = model(x)
            r_loss = F.mse_loss(recon, x)
            k_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
            p_loss = F.mse_loss(pred_y, y)
            (r_loss + 0.05 * k_loss + 1.0 * p_loss).backward()
            optimizer.step()

        if epoch % 10 == 0 or epoch == 1:
            model.eval()
            all_preds, all_gt, wt_baselines, nn_matched_dgs = [], [], [], []

            # --- PHASE 1: Metrics (No Grad) ---
            with torch.no_grad():
                for batch in val_loader:
                    emb, lbl = batch['emb'].to(device), batch['label_scaled'].to(device)
                    _, pred_y, _, _ = model(emb)
                    all_preds.extend(train_ds.scaler.inverse_transform(pred_y.cpu().numpy()).flatten())
                    all_gt.extend(train_ds.scaler.inverse_transform(lbl.cpu().numpy()).flatten())

            # --- PHASE 2: Discovery (Requires Grad for optimize_latent) ---
            batch_cluster_ids = batch['cluster_id']
            unique_clusters = np.unique(batch_cluster_ids)

            cluster_cache = {}
            with torch.no_grad():
                for c_id in unique_clusters:
                    l_idxs = val_ds.cluster_to_indices[c_id]
                    l_embs = val_ds.embeddings[l_idxs].to(device)

                    h_lib = model.encoder(l_embs)
                    recon_lib = model.decoder(h_lib[:, :model.latent_dim])

                    cluster_cache[c_id] = {
                        'recon': recon_lib,
                        'labels': val_ds.raw_labels[l_idxs]
                    }

            # 3. Optimize Latents (This is still slow because of backprop, but necessary for your logic)
            wt_dg, wt_emb = batch['wt_dg'].to(device), batch['wt_emb'].to(device)
            z_dream = optimize_latent(model, wt_emb)

            # 4. Perform Matching (Vectorized)
            with torch.no_grad():
                dream_recons = model.decoder(z_dream)

                for i in range(len(batch_cluster_ids)):
                    c_id = batch_cluster_ids[i]

                    lib_recon = cluster_cache[c_id]['recon']
                    lib_labels = cluster_cache[c_id]['labels']

                    dist = torch.norm(lib_recon - dream_recons[i].unsqueeze(0), dim=1, p=2)

                    nn_idx = torch.argmin(dist).item()

                    wt_baselines.append(wt_dg[i].item())
                    nn_matched_dgs.append(lib_labels[nn_idx].item())

            # --- Reporting ---
            all_preds, all_gt = np.array(all_preds), np.array(all_gt)
            wt_baselines, nn_matched_dgs = np.array(wt_baselines), np.array(nn_matched_dgs)

            mae = mean_absolute_error(all_gt, all_preds)
            rmse = np.sqrt(mean_squared_error(all_gt, all_preds))
            r2 = r2_score(all_gt, all_preds)
            rho, _ = spearmanr(all_gt, all_preds)
            pear, _ = pearsonr(all_gt, all_preds)

            success = np.sum(nn_matched_dgs < wt_baselines)

            print(f"\n--- Epoch {epoch:03d} ---")
            print(f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")
            print(f"Spearman: {rho:.4f} | Pearson: {pear:.4f}")
            print(f"Reconstructed Matching Success: {success}/{len(wt_baselines)} proteins better than WT.")

## Running the model
(Hyperparameter optimisation done here maybe)

In [ ]:
train_and_validate(train_path, val_path)

## Loss-Weight Ablation (Five-Value Sweeps + Combined 3-Metric Figure)

This section performs one-factor ablations for the VAE loss weights. For each sweep, 5 values are tested while the other two weights are fixed at baseline.

- KL_WEIGHT: [0.0, 0.01, 0.03, 0.07, 0.15]
- PREDICTION_WEIGHT: [0.25, 0.5, 1.0, 1.5, 2.5]
- RECONSTRUCTION_WEIGHT: [0.25, 0.5, 1.0, 1.5, 2.0]

To support error bars, each sweep value is run across multiple seeds and aggregated into mean and standard deviation.

Outputs:
- `results/vae_ablation/loss_weight_ablation_runs.csv`
- `results/vae_ablation/loss_weight_ablation_runs.json`
- `results/vae_ablation/loss_weight_ablation_summary.csv`
- `results/vae_ablation/loss_weight_ablation_summary.json`
- `results/vae_ablation/loss_weight_ablation_combined_3metrics_scatter_errorbars.png`

Figure layout:
- 3 panels: RMSE, R2, Spearman
- Each panel overlays KL_WEIGHT, PREDICTION_WEIGHT, and RECONSTRUCTION_WEIGHT
- Visual encoding uses both color and marker shape differences

Note: `FAST_MODE=True` keeps runtime practical for iterative experimentation.

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


# -----------------------------
# Repro + runtime controls
# -----------------------------
BASE_SEED = 42
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

FAST_MODE = True
FAST_TRAIN_MAX = 12000
FAST_VAL_MAX = 4000
EPOCHS = 15 if FAST_MODE else 60
PATIENCE = 4
REPEATS_PER_VALUE = 3

# Keep latent optimization aligned with the notebook defaults
AB_OPTIM_STEPS = 25
AB_OPTIM_LR = 0.01
LATENT_PENALTY = 0.05

# Fallback to local processed paths if train_path/val_path are not set
if 'train_path' not in globals():
    train_path = Path('data/processed/train_sampled_esm2_650m.pt')
if 'val_path' not in globals():
    val_path = Path('data/processed/val_full_esm2_650m.pt')

if not Path(train_path).exists() or not Path(val_path).exists():
    raise FileNotFoundError(f"Missing .pt files: train={train_path}, val={val_path}")


def _set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _subset_dataset(ds, max_rows, rng_seed):
    if max_rows is None or len(ds) <= max_rows:
        return ds
    idx = np.random.RandomState(rng_seed).choice(len(ds), size=max_rows, replace=False)
    emb = ds.embeddings[idx]
    y_scaled = ds.labels_scaled[idx].numpy()
    y_raw = ds.raw_labels[idx].numpy()
    clusters = ds.clusters[idx]
    mut_types = ds.mut_types[idx]

    # Build a lightweight dataset-like object
    class _DS(torch.utils.data.Dataset):
        def __init__(self):
            self.embeddings = emb
            self.labels_scaled = torch.from_numpy(y_scaled).float()
            self.raw_labels = torch.from_numpy(y_raw).float()
            self.clusters = clusters
            self.mut_types = mut_types
            self.scaler = ds.scaler

            self.cluster_to_indices = {}
            for i, c in enumerate(self.clusters):
                self.cluster_to_indices.setdefault(c, []).append(i)

            self.wt_lookup = {}
            for i in range(len(self.clusters)):
                if str(self.mut_types[i]).lower().strip() == 'wt':
                    self.wt_lookup[self.clusters[i]] = {
                        'dg': float(self.raw_labels[i].item()),
                        'emb': self.embeddings[i],
                        'idx': i,
                    }

        def __len__(self):
            return self.embeddings.shape[0]

        def __getitem__(self, i):
            c = self.clusters[i]
            wt_info = self.wt_lookup.get(c, None)
            wt_dg = wt_info['dg'] if wt_info else float(self.raw_labels[i].item())
            wt_emb = wt_info['emb'] if wt_info else self.embeddings[i]
            return {
                'emb': self.embeddings[i],
                'label_scaled': self.labels_scaled[i],
                'wt_dg': torch.tensor([wt_dg]).float(),
                'wt_emb': wt_emb,
                'cluster_id': c,
            }

    return _DS()


def evaluate_config(
    kl_weight,
    prediction_weight,
    reconstruction_weight,
    seed,
    epochs=EPOCHS,
    batch_size=64,
    learning_rate=1e-4,
    optim_steps=AB_OPTIM_STEPS,
    optim_lr=AB_OPTIM_LR,
    latent_penalty=LATENT_PENALTY,
):
    _set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    train_ds = ProteinDataset(train_path)
    val_ds = ProteinDataset(val_path, scaler=train_ds.scaler)

    if FAST_MODE:
        train_ds = _subset_dataset(train_ds, FAST_TRAIN_MAX, rng_seed=seed)
        val_ds = _subset_dataset(val_ds, FAST_VAL_MAX, rng_seed=seed)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = ProteinVAE(input_dim=1280, latent_dim=256, hidden_dim=1024).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_mae = float('inf')
    best_state = None
    wait = 0

    for _epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            x = batch['emb'].to(device)
            y = batch['label_scaled'].to(device)

            optimizer.zero_grad()
            recon, pred_y, mu, logvar = model(x)
            r_loss = F.mse_loss(recon, x)
            k_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
            p_loss = F.mse_loss(pred_y, y)
            total_loss = (
                reconstruction_weight * r_loss
                + kl_weight * k_loss
                + prediction_weight * p_loss
            )
            total_loss.backward()
            optimizer.step()

        # Validation for early stopping
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                emb = batch['emb'].to(device)
                lbl = batch['label_scaled'].to(device)
                _, pred_y, _, _ = model(emb)
                val_preds.extend(train_ds.scaler.inverse_transform(pred_y.cpu().numpy()).flatten())
                val_true.extend(train_ds.scaler.inverse_transform(lbl.cpu().numpy()).flatten())

        val_mae = float(mean_absolute_error(val_true, val_preds))
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    # Final predictive metrics
    all_preds, all_gt = [], []
    with torch.no_grad():
        for batch in val_loader:
            emb = batch['emb'].to(device)
            lbl = batch['label_scaled'].to(device)
            _, pred_y, _, _ = model(emb)
            all_preds.extend(train_ds.scaler.inverse_transform(pred_y.cpu().numpy()).flatten())
            all_gt.extend(train_ds.scaler.inverse_transform(lbl.cpu().numpy()).flatten())

    all_preds = np.array(all_preds)
    all_gt = np.array(all_gt)

    mae_v = float(mean_absolute_error(all_gt, all_preds))
    rmse_v = float(np.sqrt(mean_squared_error(all_gt, all_preds)))
    r2_v = float(r2_score(all_gt, all_preds))
    spearman_v = float(spearmanr(all_gt, all_preds)[0])
    pearson_v = float(pearsonr(all_gt, all_preds)[0])

    return {
        'MAE': mae_v,
        'RMSE': rmse_v,
        'R2': r2_v,
        'Spearman': spearman_v,
        'Pearson': pearson_v,
        'n_val_used': int(len(all_gt)),
    }


# -----------------------------
# One-factor sweeps (5 values each, as requested)
# -----------------------------
base = {
    'KL_WEIGHT': 0.05,
    'PREDICTION_WEIGHT': 1.0,
    'RECONSTRUCTION_WEIGHT': 1.0,
}

kl_values = [0.0, 0.01, 0.03, 0.07, 0.15]
pred_values = [0.25, 0.5, 1.0, 1.5, 2.5]
recon_values = [0.25, 0.5, 1.0, 1.5, 2.0]

sweeps = [
    ('KL_WEIGHT', kl_values),
    ('PREDICTION_WEIGHT', pred_values),
    ('RECONSTRUCTION_WEIGHT', recon_values),
]

runs = []
for sweep_name, values in sweeps:
    print(f"\n=== Running sweep: {sweep_name} ({len(values)} values x {REPEATS_PER_VALUE} repeats) ===")
    for v in values:
        for rep in range(REPEATS_PER_VALUE):
            seed = BASE_SEED + rep
            cfg = dict(base)
            cfg[sweep_name] = v

            out = evaluate_config(
                kl_weight=cfg['KL_WEIGHT'],
                prediction_weight=cfg['PREDICTION_WEIGHT'],
                reconstruction_weight=cfg['RECONSTRUCTION_WEIGHT'],
                seed=seed,
            )

            runs.append(
                {
                    'sweep': sweep_name,
                    'sweep_value': float(v),
                    'repeat': rep,
                    'seed': seed,
                    **cfg,
                    **out,
                }
            )
            print(
                f"{sweep_name}={v} | rep={rep} | "
                f"MAE={out['MAE']:.4f}, RMSE={out['RMSE']:.4f}, R2={out['R2']:.4f}, "
                f"Spearman={out['Spearman']:.4f}, Pearson={out['Pearson']:.4f}"
            )

runs_df = pd.DataFrame(runs)
metrics = ['RMSE', 'R2', 'Spearman']

# Aggregate mean/std for error bars
agg_spec = {}
for m in metrics:
    agg_spec[f'{m}_mean'] = (m, 'mean')
    agg_spec[f'{m}_std'] = (m, 'std')
summary_df = (
    runs_df
    .groupby(['sweep', 'sweep_value'], as_index=False)
    .agg(**agg_spec)
    .sort_values(['sweep', 'sweep_value'])
)
summary_df = summary_df.fillna(0.0)

# Resolve output dir to project-root/results/vae_ablation when possible
train_path_resolved = Path(train_path).resolve()
if len(train_path_resolved.parents) >= 3 and train_path_resolved.parents[1].name == 'data':
    repo_root = train_path_resolved.parents[2]
else:
    repo_root = Path.cwd()
out_dir = repo_root / 'results' / 'vae_ablation'
out_dir.mkdir(parents=True, exist_ok=True)

runs_csv_path = out_dir / 'loss_weight_ablation_runs.csv'
runs_json_path = out_dir / 'loss_weight_ablation_runs.json'
summary_csv_path = out_dir / 'loss_weight_ablation_summary.csv'
summary_json_path = out_dir / 'loss_weight_ablation_summary.json'
fig_path = out_dir / 'loss_weight_ablation_combined_3metrics_scatter_errorbars.png'

runs_df.to_csv(runs_csv_path, index=False)
summary_df.to_csv(summary_csv_path, index=False)

with open(runs_json_path, 'w', encoding='utf-8') as f:
    json.dump(runs, f, indent=2)
with open(summary_json_path, 'w', encoding='utf-8') as f:
    json.dump(summary_df.to_dict(orient='records'), f, indent=2)

print('Saved:', runs_csv_path)
print('Saved:', runs_json_path)
print('Saved:', summary_csv_path)
print('Saved:', summary_json_path)

# -----------------------------
# Combined figure: 3 panels (RMSE, R2, Spearman)
# Each panel overlays KL/Prediction/Reconstruction with color + marker shape
# -----------------------------
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
})

row_order = ['KL_WEIGHT', 'PREDICTION_WEIGHT', 'RECONSTRUCTION_WEIGHT']
label_map = {
    'KL_WEIGHT': 'KL Weight',
    'PREDICTION_WEIGHT': 'Prediction Weight',
    'RECONSTRUCTION_WEIGHT': 'Reconstruction Weight',
}
color_map = {
    'KL_WEIGHT': '#1f77b4',
    'PREDICTION_WEIGHT': '#d62728',
    'RECONSTRUCTION_WEIGHT': '#2ca02c',
}
marker_map = {
    'KL_WEIGHT': '^',
    'PREDICTION_WEIGHT': 's',
    'RECONSTRUCTION_WEIGHT': 'o',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=260, sharex=False)

for c, metric in enumerate(metrics):
    ax = axes[c]
    for sweep_name in row_order:
        sub = summary_df[summary_df['sweep'] == sweep_name].sort_values('sweep_value')
        x = sub['sweep_value'].values
        y = sub[f'{metric}_mean'].values
        yerr = sub[f'{metric}_std'].values

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            fmt=marker_map[sweep_name],
            linestyle='none',
            color=color_map[sweep_name],
            ecolor=color_map[sweep_name],
            elinewidth=1.2,
            capsize=3,
            markersize=6,
            alpha=0.95,
            label=label_map[sweep_name],
        )

    ax.set_title(metric)
    ax.set_xlabel('Weight value')
    if c == 0:
        ax.set_ylabel('Metric value')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.04))
fig.suptitle('ProteinVAE Loss-Weight Ablation: Combined RMSE, R2, and Spearman', y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(fig_path, bbox_inches='tight')
plt.show()

print('Saved:', fig_path)

summary_df